## Merging

## Cleaning

In [ ]:
import pandas as pd
from pathlib import Path
import re

RAW_DIR = Path("raw")
CLEAN_DIR = Path("cleaned")

CLEAN_DIR.mkdir(exist_ok=True)

def read_weather_csv(path):
    """Read CSV with encoding fallback"""
    encodings = ["utf-8", "cp1252", "latin1"]

    for enc in encodings:
        try:
            df = pd.read_csv(
                path,
                skiprows=5,
                encoding=enc,
                engine="python"
            )
            return df
        except UnicodeDecodeError:
            continue

    raise RuntimeError(f"Could not read file: {path}")


def normalize_column(name):
    """
    Normalize column names
    Example:
    'Temperature (°C)' -> temperature_c
    'Wind Speed (km/h)' -> wind_speed_km_h
    """

    name = str(name)

    name = name.replace("°", "deg")
    name = name.replace("%", "pct")

    name = name.lower()

    name = re.sub(r"[()]", "", name)
    name = re.sub(r"[^a-z0-9]+", "_", name)

    name = re.sub(r"_+", "_", name)

    return name.strip("_")


def clean_dataset(file_path):

    print(f"Processing {file_path.name}")

    df = read_weather_csv(file_path)

    # Normalize column names
    df.columns = [normalize_column(c) for c in df.columns]

    # Drop empty rows
    df = df.dropna(how="all")

    df = df.reset_index(drop=True)

    return df


cleaned_frames = []

for csv_file in sorted(RAW_DIR.glob("*.csv")):

    df = clean_dataset(csv_file)

    cleaned_frames.append(df)

    output_path = CLEAN_DIR / f"{csv_file.stem}_cleaned.csv"

    df.to_csv(output_path, index=False, encoding="utf-8")

    print(f"Saved cleaned file -> {output_path}\n")


print("Merging yearly datasets...")

merged = pd.concat(cleaned_frames, ignore_index=True)


# Try to detect datetime column automatically
datetime_candidates = [c for c in merged.columns if "time" in c or "date" in c]

if datetime_candidates:
    dt_col = datetime_candidates[0]

    merged[dt_col] = pd.to_datetime(merged[dt_col], errors="coerce")

    merged = merged.sort_values(dt_col)

    merged = merged.drop_duplicates(subset=[dt_col])


merged = merged.reset_index(drop=True)


merged_path = CLEAN_DIR / "kiit_weather_merged.csv"

merged.to_csv(
    merged_path,
    index=False,
    encoding="utf-8"
)

print(f"Merged dataset saved -> {merged_path}")

print(f"Final merged shape: {merged.shape}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv("./cleaned/kiit_weather_merged.csv")

# Convert '--' to NaN
df.replace("--", np.nan, inplace=True)

# Parse datetime
df["date_time"] = pd.to_datetime(df["date_time"])

# Sort by time
df = df.sort_values("date_time").reset_index(drop=True)

print("Dataset Shape:", df.shape)
print("\nTotal Missing Values:", df.isna().sum().sum())

# Missing values per column
missing_counts = df.isna().sum()
missing_percent = (missing_counts / len(df)) * 100

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percent": missing_percent
}).sort_values("missing_percent", ascending=False)

print("\nMissing Value Summary:")
display(missing_summary[missing_summary["missing_count"] > 0])

# Detect position of missing values (start, end, random)
def missing_position(series):
    if series.isna().all():
        return "All Missing"
    
    first_valid = series.first_valid_index()
    last_valid = series.last_valid_index()
    
    if first_valid is None:
        return "All Missing"
    
    start_missing = series.iloc[:first_valid].isna().all()
    end_missing = series.iloc[last_valid+1:].isna().all()
    
    if start_missing and not end_missing:
        return "Missing at Beginning"
    elif end_missing and not start_missing:
        return "Missing at End"
    elif start_missing and end_missing:
        return "Missing at Beginning & End"
    elif series.isna().any():
        return "Random Missing"
    else:
        return "No Missing"

missing_pattern = {col: missing_position(df[col]) for col in df.columns if col != "date_time"}

pattern_df = pd.DataFrame.from_dict(missing_pattern, orient="index", columns=["missing_pattern"])
print("\nMissing Pattern:")
display(pattern_df)

# Heatmap of missing values over time
plt.figure(figsize=(18,6))
sns.heatmap(df.set_index("date_time").isna(),
            cbar=False,
            yticklabels=False)

plt.title("Missing Values Heatmap (Time Series)")
plt.xlabel("Variables")
plt.ylabel("Time")
plt.show()

In [ ]:
print("Total rows:", len(df))

print("Start timestamp:", df["date_time"].min())
print("End timestamp:", df["date_time"].max())

print("Unique timestamps:", df["date_time"].nunique())

In [ ]:
duplicates = df[df["date_time"].duplicated()]

print("Duplicate timestamps:", len(duplicates))

display(duplicates.head())

In [ ]:
time_diffs = df["date_time"].diff()

display(time_diffs.value_counts().head(10))

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("./cleaned/kiit_weather_merged.csv")

df.replace("--", np.nan, inplace=True)
df["date_time"] = pd.to_datetime(df["date_time"])
df = df.sort_values("date_time").reset_index(drop=True)

segments = []

for col in df.columns:
    
    if col == "date_time":
        continue

    series = pd.to_numeric(df[col], errors="coerce")
    valid_idx = series.dropna().index

    if len(valid_idx) < 20:
        continue

    gaps = np.diff(valid_idx)

    start_i = 0

    for i in range(1, len(gaps)):
        
        if gaps[i] != gaps[i-1]:

            segments.append({
                "column": col,
                "start_row": valid_idx[start_i],
                "end_row": valid_idx[i],
                "start_time": df.loc[valid_idx[start_i],"date_time"],
                "end_time": df.loc[valid_idx[i],"date_time"],
                "periodicity": gaps[i-1],
                "segment_length": i-start_i
            })

            start_i = i

    segments.append({
        "column": col,
        "start_row": valid_idx[start_i],
        "end_row": valid_idx[-1],
        "start_time": df.loc[valid_idx[start_i],"date_time"],
        "end_time": df.loc[valid_idx[-1],"date_time"],
        "periodicity": gaps[-1],
        "segment_length": len(valid_idx)-start_i
    })

segments_df = pd.DataFrame(segments)

segments_df = segments_df.sort_values(["column","start_time"])

segments_df.to_csv("sensor_periodicity_segments.csv", index=False)

display(segments_df)

In [ ]:
# Add year and month columns
df["year"] = df["date_time"].dt.year
df["month"] = df["date_time"].dt.month

years = sorted(df["year"].unique())

for year in years:
    
    df_year = df[df["year"] == year]
    
    # Compute monthly missing fraction
    monthly_missing = (
        df_year
        .groupby("month")
        .apply(lambda x: x.isna().mean())
    )
    
    # Remove helper columns
    monthly_missing = monthly_missing.drop(columns=["year","month"], errors="ignore")
    
    plt.figure(figsize=(18,6))
    
    sns.heatmap(
        monthly_missing,
        cmap="rocket_r",
        cbar_kws={"label": "Missing Fraction"}
    )
    
    plt.title(f"Monthly Missing Value Heatmap — {year}")
    plt.xlabel("Variables")
    plt.ylabel("Month")
    
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

# ----------------------------
# Load dataset
# ----------------------------
df = pd.read_csv("./cleaned/kiit_weather_merged.csv")

df.replace("--", np.nan, inplace=True)

df["date_time"] = pd.to_datetime(df["date_time"])
df = df.sort_values("date_time").reset_index(drop=True)

# ----------------------------
# Trim dataset
# ----------------------------
df = df[df["date_time"] >= "2024-08-01"].reset_index(drop=True)

# ----------------------------
# Drop useless features
# ----------------------------
drop_cols = [
    "prevailing_wind_direction",
    "high_wind_direction",

    "temp_degc_1","high_temp_degc_1","low_temp_degc_1",
    "hum_pct_1","high_hum_pct_1","low_hum_pct_1",
    "dew_point_degc_1","high_dew_point_degc_1","low_dew_point_degc_1",
    "wet_bulb_degc_1","high_wet_bulb_degc_1","low_wet_bulb_degc_1",
    "heat_index_degc_1","high_heat_index_degc_1"
]

df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# ----------------------------
# Convert numeric columns
# ----------------------------
for col in df.columns:
    if col != "date_time":
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ----------------------------
# Set index
# ----------------------------
df = df.set_index("date_time")

# ----------------------------
# Resample to 15-minute resolution
# ----------------------------
df = df.resample("15min").mean()

# ----------------------------
# Save original missing mask
# ----------------------------
original_missing = df.isna()

# ----------------------------
# Interpolate
# ----------------------------
df_interp = df.interpolate(method="time")
df_interp = df_interp.ffill().bfill()

# ----------------------------
# Detect interpolated values
# ----------------------------
interpolated_mask = original_missing & df_interp.notna()

# ----------------------------
# Visualization matrix
# ----------------------------
viz_matrix = np.zeros(df_interp.shape)

viz_matrix[df_interp.isna()] = 1
viz_matrix[interpolated_mask.values] = 2

# ----------------------------
# Color map
# ----------------------------
cmap = ListedColormap([
    "#0b0c2a",   # original
    "#ffffff",   # missing
    "#00ff00"    # interpolated
])

# ----------------------------
# Heatmap
# ----------------------------
plt.figure(figsize=(18,6))

sns.heatmap(
    viz_matrix,
    cmap=cmap,
    cbar=False,
    yticklabels=False,
    xticklabels=df_interp.columns
)

plt.title("Missing & Interpolated Values Heatmap (15-min resolution)")
plt.xlabel("Variables")
plt.ylabel("Time")

plt.show()

In [ ]:
# Reset index so date_time becomes a column again
df_clean = df_interp.reset_index()

# Save cleaned dataset
df_clean.to_csv("./cleaned/kiit_weather_cleaned.csv", index=False)

print("Cleaned dataset saved to ./cleaned/kiit_weather_cleaned.csv")
print("Final shape:", df_clean.shape)

## EDA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("./cleaned/kiit_weather_cleaned.csv")

df["date_time"] = pd.to_datetime(df["date_time"])

df = df.sort_values("date_time")

df.set_index("date_time", inplace=True)

print("Dataset shape:", df.shape)

display(df.head())
display(df.describe())

In [ ]:
print("Remaining missing values:", df.isna().sum().sum())

In [ ]:
print("Start:", df.index.min())
print("End:", df.index.max())

print("Total observations:", len(df))

In [ ]:
df.index.to_series().diff().value_counts().head()

In [ ]:
key_vars = [
    "aqi",
    "pm_2_5_ug_m",
    "pm_10_ug_m",
    "temp_degc",
    "hum_pct",
    "avg_wind_speed_km_h"
]

df[key_vars].hist(figsize=(12,8), bins=40)

plt.suptitle("Distribution of Key Variables")
plt.show()

### Distribution of Key Environmental Variables

The distribution plots of the primary environmental variables provide insight into the statistical characteristics and variability of the dataset after preprocessing. These histograms help identify the typical value ranges, skewness, and potential outliers in the variables that are most relevant for **AQI and temperature prediction**.

The histogram of **Air Quality Index (AQI)** shows that the majority of observations fall roughly between **70 and 180**, with the highest concentration around the **100–150 range**. This indicates that air quality conditions in the dataset are most frequently in the **moderate to unhealthy-for-sensitive-groups categories**. The distribution is moderately right-skewed, with fewer occurrences of very high AQI values beyond 200. Such a pattern suggests that extreme pollution events are relatively rare but still present, which is typical for urban atmospheric datasets where pollution spikes occur during certain meteorological conditions such as stagnant air or increased emissions.

The distribution of **PM2.5 concentrations** exhibits a strong **right-skewed pattern**, where a large number of observations are clustered at lower concentration levels while progressively fewer observations occur at higher values. This is characteristic of particulate pollution data, where most time periods experience relatively low to moderate particulate levels, while occasional pollution episodes cause large spikes. The long tail extending toward higher values indicates the presence of intermittent high pollution events, which significantly influence AQI values since PM2.5 is one of the dominant contributors to AQI calculations.

A similar pattern can be observed in the **PM10 distribution**, which also shows a right-skewed shape with most values concentrated below approximately **120 µg/m³**. Like PM2.5, PM10 concentrations tend to remain relatively low during normal atmospheric conditions but can increase during events such as dust transport, construction activity, or reduced atmospheric dispersion. The similarity between the PM2.5 and PM10 distributions further supports the expectation that particulate pollutants are strongly correlated and play a major role in determining overall air quality.

The histogram of **temperature (temp_degc)** reveals a roughly **bell-shaped distribution** centered around **26–29°C**, which suggests that the dataset predominantly captures warm climatic conditions. This is consistent with the geographical and seasonal context of the data, where temperatures tend to remain within a moderate to warm range for most of the year. The relatively symmetric distribution indicates that temperature varies gradually over time without extreme fluctuations, making it a stable environmental variable for predictive modeling.

In contrast, the distribution of **relative humidity (hum_pct)** shows a noticeable skew toward higher humidity levels, with most observations concentrated between **70% and 95%**. This indicates that the dataset represents a predominantly humid environment, which is typical for coastal or monsoon-influenced climates. High humidity levels are particularly relevant in air quality analysis because they influence particulate formation, aerosol chemistry, and pollutant dispersion dynamics.

Finally, the distribution of **average wind speed (avg_wind_speed_km_h)** shows that most wind speeds fall within a relatively low range, typically between **2 and 10 km/h**, with fewer occurrences of stronger winds. The right-skewed distribution indicates that calm or low-wind conditions are common, while strong wind events are comparatively rare. Wind speed is a critical meteorological factor affecting air quality, as stronger winds promote pollutant dispersion, while low wind speeds can lead to pollutant accumulation near the surface.

Overall, these distribution plots highlight the statistical properties of the environmental variables used in the dataset. The presence of right-skewed pollutant distributions, relatively stable temperature patterns, high humidity conditions, and generally low wind speeds suggests a typical urban atmospheric environment where pollution levels are influenced by both emission sources and meteorological conditions. Understanding these distributions is important for selecting appropriate modeling approaches and transformations during the time-series prediction stage.


In [ ]:
plt.figure(figsize=(15,5))
plt.plot(df.index, df["aqi"])
plt.title("AQI Time Series")
plt.show()


plt.figure(figsize=(15,5))
plt.plot(df.index, df["temp_degc"])
plt.title("Temperature Time Series")
plt.show()

### Temporal Behavior of AQI and Temperature

The time-series plots of **AQI** and **temperature** provide insight into the temporal dynamics of air quality and meteorological conditions across the observation period. These plots allow us to examine long-term trends, seasonal variations, and short-term fluctuations that are important for building predictive time-series models.

The **AQI time series** demonstrates significant variability over time, with both gradual trends and abrupt spikes. For most of the observed period, AQI values fluctuate between approximately **50 and 200**, indicating moderate to occasionally unhealthy air quality conditions. However, several sharp peaks exceeding **300 and even reaching above 600** are visible, representing episodic pollution events. These spikes likely correspond to periods of unfavorable atmospheric conditions such as low wind speeds, increased emissions, or thermal inversions that trap pollutants near the surface. Additionally, the AQI series exhibits noticeable seasonal variation. Higher pollution levels appear during certain months, particularly toward the late autumn and winter periods, while comparatively lower AQI values are observed during mid-year months. Such seasonal patterns are commonly associated with changes in atmospheric stability, rainfall patterns, and regional emission activities. The presence of both short-term fluctuations and seasonal trends suggests that AQI is influenced by a combination of meteorological conditions and environmental factors, making it well-suited for time-series modeling approaches that capture temporal dependencies.

The **temperature time series**, on the other hand, displays a much smoother and more structured pattern compared to AQI. Temperature values generally range between approximately **15°C and 40°C**, with most observations clustered between **25°C and 32°C**. Unlike AQI, temperature exhibits a clear **seasonal cycle** rather than abrupt spikes. The plot shows gradual warming and cooling phases that correspond to seasonal transitions throughout the year. For example, lower temperatures appear during winter months, while higher temperatures occur during late spring and early summer. These gradual transitions indicate strong temporal continuity, meaning that temperature values at a given time are highly dependent on recent observations. This smooth and predictable pattern is typical of meteorological variables and makes temperature particularly amenable to forecasting models that exploit temporal autocorrelation.

Comparing the two series highlights an important distinction in their temporal behavior. While temperature evolves gradually and follows relatively predictable seasonal cycles, AQI exhibits more irregular and volatile dynamics driven by both environmental conditions and episodic pollution events. Nevertheless, certain patterns in AQI appear to coincide with broader seasonal trends, suggesting that meteorological variables such as temperature, humidity, and wind speed may influence air quality levels. Understanding these temporal characteristics is essential for designing effective forecasting models, as it indicates that AQI prediction may require models capable of capturing both **short-term fluctuations and longer seasonal patterns**, whereas temperature prediction primarily relies on modeling smoother and more continuous temporal trends.


In [ ]:
df["hour"] = df.index.hour

hourly_pattern = df.groupby("hour")[["aqi","temp_degc"]].mean()

hourly_pattern.plot(figsize=(10,5))

plt.title("Average Daily Pattern")
plt.show()

### Average Daily Pattern of AQI and Temperature

The plot illustrating the **average daily pattern** represents the mean values of AQI and temperature aggregated by hour of the day across the entire dataset. By grouping the data by the hour component of the timestamp and averaging the observations, this visualization highlights the **typical diurnal cycle** of both variables. Such patterns are important in environmental time-series analysis because many atmospheric processes exhibit strong daily periodicity driven by solar radiation, human activity cycles, and atmospheric mixing processes.

The **AQI curve** shows noticeable variation throughout the day, with relatively higher values during the early morning hours and again during the late evening and nighttime periods. AQI levels appear to decrease gradually during the daytime, reaching their lowest levels in the mid-afternoon. This pattern is consistent with well-known atmospheric dynamics. During the daytime, increased solar heating causes stronger vertical air mixing in the atmosphere, which helps disperse pollutants and reduces their concentration near the surface. In contrast, during nighttime and early morning hours, the atmosphere tends to become more stable with weaker air circulation, allowing pollutants to accumulate closer to ground level. Additionally, human activity patterns such as morning and evening traffic can contribute to higher pollutant concentrations during these periods.

The **temperature curve**, in contrast, exhibits a typical diurnal temperature cycle. Temperatures are lowest during the early morning hours, gradually increasing throughout the morning as solar radiation intensifies. The highest average temperatures occur around early to mid-afternoon, after which temperatures begin to decline slowly as solar heating decreases toward evening and nighttime. This smooth rise and fall in temperature reflects the natural daily heating and cooling cycle driven by the sun’s radiation and the thermal response of the Earth's surface.

When comparing the two curves, an inverse relationship between AQI and temperature can be observed during certain periods of the day. As temperature increases toward midday and early afternoon, AQI tends to decline, suggesting that stronger atmospheric mixing during warmer conditions helps disperse pollutants. Conversely, during cooler nighttime conditions, pollutant concentrations increase due to reduced atmospheric turbulence and pollutant dispersion. This daily pattern highlights the strong influence of meteorological conditions on air quality dynamics and suggests that incorporating time-of-day features and temperature-related variables may improve the performance of predictive models for AQI forecasting.


In [ ]:
df["month"] = df.index.month

monthly_avg = df.groupby("month")[["aqi","temp_degc"]].mean()

monthly_avg.plot(figsize=(10,5))

plt.title("Monthly Average Trends")
plt.show()

### Monthly Average Trends of AQI and Temperature

The monthly average trends plot illustrates the seasonal variation in both **AQI** and **temperature** across the dataset. By aggregating observations by month and computing their mean values, this visualization highlights broader seasonal patterns that may not be easily visible in short-term time-series plots. Understanding these seasonal cycles is essential for air quality forecasting because atmospheric conditions, emission patterns, and meteorological processes vary significantly throughout the year.

The **AQI trend** shows a pronounced seasonal pattern. Higher AQI values are observed during the beginning and end of the year, with the highest averages occurring around **January and December**, where AQI values approach approximately **180–200**. Following this peak, AQI gradually decreases through the early months of the year, reaching its lowest levels around **June and July**, where average values fall to roughly **65–70**. After mid-year, AQI begins to increase again through the later months. This pattern is consistent with typical seasonal air quality dynamics. During winter months, atmospheric conditions often become more stable, reducing vertical mixing and allowing pollutants to accumulate near the surface. Additionally, increased human activities such as heating, industrial processes, and traffic may contribute to higher emission levels. In contrast, during mid-year months, stronger solar heating, increased wind circulation, and in many regions the presence of rainfall contribute to improved atmospheric dispersion and pollutant removal, resulting in lower AQI values.

The **temperature trend** follows a more gradual and smooth seasonal cycle. Temperatures increase steadily from the early months of the year, reaching a peak during the late spring and early summer months around **April and May**, where the average temperature approaches approximately **30°C**. After this peak, temperatures remain relatively stable for several months before gradually decreasing toward the end of the year. This seasonal progression reflects the natural annual heating cycle driven by solar radiation and climatic conditions.

Comparing the two curves reveals an inverse relationship between AQI and temperature across seasons. As temperatures rise during the middle of the year, AQI levels generally decrease, suggesting that warmer conditions may enhance atmospheric mixing and pollutant dispersion. Conversely, during cooler months when atmospheric stability increases, pollutant accumulation becomes more likely, leading to higher AQI values. These seasonal patterns emphasize the strong influence of meteorological factors on air quality and indicate that incorporating **seasonal features such as month or seasonal indicators** could improve the performance of predictive models. Additionally, the clear periodic structure observed in both AQI and temperature suggests that time-series forecasting models should account for **seasonal components** to capture these recurring annual trends effectively.


In [ ]:
corr = df.corr()

plt.figure(figsize=(12,10))

sns.heatmap(corr, cmap="coolwarm", center=0)

plt.title("Feature Correlation Matrix")

plt.show()

### Feature Correlation Analysis

The feature correlation matrix provides a comprehensive overview of the linear relationships between the variables present in the dataset. Each cell in the heatmap represents the **Pearson correlation coefficient** between two variables, with values ranging from **−1 to 1**. Positive correlations indicate that two variables tend to increase together, while negative correlations indicate that as one variable increases, the other tends to decrease. Visualizing these relationships helps identify strongly related variables, potential redundancies, and the key predictors that may influence AQI and temperature.

One of the most noticeable patterns in the matrix is the strong positive correlation among **temperature-related variables**, including temperature, dew point, wet bulb temperature, heat index, THW index, and THSW index. These variables are derived from similar atmospheric conditions and therefore exhibit very high correlations with each other. For example, heat index and THW index are calculated using temperature and humidity inputs, which explains why they closely follow the temperature trends. This cluster of strongly correlated features indicates that these variables contain overlapping information about thermal conditions in the atmosphere. While they may provide useful contextual signals, some of them may also be redundant in predictive models and could potentially be reduced through feature selection techniques.

Another clear pattern emerges among the **solar radiation and ultraviolet-related variables**, such as solar radiation, UV index, and UV dose. These variables are strongly correlated because they all represent different aspects of solar energy reaching the Earth's surface. Higher solar radiation typically corresponds to higher UV index values and greater UV dose accumulation. These features are particularly relevant for understanding daytime atmospheric conditions, as solar radiation influences atmospheric heating, convection, and photochemical reactions that can affect pollutant formation.

The matrix also highlights strong correlations between **particulate matter measurements**, specifically PM1, PM2.5, and PM10. These pollutants originate from similar emission sources and atmospheric processes, which explains why their concentrations tend to rise and fall together. Because particulate matter levels directly influence AQI calculations, the strong correlation between these variables confirms their central role in determining overall air quality conditions. This relationship suggests that particulate measurements will likely be among the most important predictors for AQI forecasting models.

Meteorological variables such as **wind speed and barometric pressure** show weaker correlations with most other variables. This is expected because these factors influence pollutant dispersion rather than directly contributing to pollutant formation. For instance, higher wind speeds can dilute pollutant concentrations by increasing atmospheric mixing, but this effect may not always appear as a strong linear correlation in aggregated datasets.

The correlation matrix also reveals a moderate relationship between **seasonal indicators and environmental variables**, represented by the inclusion of the month variable. This suggests that certain environmental conditions follow seasonal cycles, reinforcing earlier observations from the monthly trend analysis. Seasonal variation plays an important role in shaping both temperature and air quality dynamics, as factors such as solar radiation, rainfall, and atmospheric circulation patterns change throughout the year.

Overall, the correlation analysis highlights several key insights about the dataset. Environmental variables tend to cluster into groups representing **thermal conditions, solar radiation effects, and particulate pollution**, each contributing differently to the atmospheric system. The strong relationships among derived meteorological indices suggest potential redundancy, while the strong correlations among particulate matter variables confirm their central role in determining AQI. These findings are valuable for guiding **feature selection and model design**, as they help identify the most informative variables while avoiding unnecessary duplication of highly correlated features.


In [ ]:
aqi_corr = corr["aqi"].sort_values(ascending=False)

display(aqi_corr)

In [ ]:
from pandas.plotting import lag_plot

lag_plot(df["aqi"])

plt.title("AQI Lag Plot")

plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plot_acf(df["aqi"], lags=96)
plt.show()

plot_pacf(df["aqi"], lags=96)
plt.show()

In [ ]:
plt.figure(figsize=(6,6))

sns.scatterplot(x=df["temp_degc"], y=df["aqi"], alpha=0.3)

plt.title("Temperature vs AQI")

plt.show()

In [ ]:
plt.figure(figsize=(6,6))

sns.scatterplot(x=df["avg_wind_speed_km_h"], y=df["aqi"], alpha=0.3)

plt.title("Wind Speed vs AQI")

plt.show()

In [ ]:
df["aqi_rolling"] = df["aqi"].rolling(96).mean()

plt.figure(figsize=(15,5))

plt.plot(df["aqi"], alpha=0.4)
plt.plot(df["aqi_rolling"], color="red")

plt.title("AQI Rolling Mean (Daily)")
plt.show()

### AQI Rolling Mean Trend Analysis

The rolling mean plot of AQI provides a smoothed representation of air quality trends over time by averaging AQI values within a moving window of observations. In this visualization, the raw AQI values are displayed as a light blue series, while the red line represents the **daily rolling mean**, which helps reduce short-term fluctuations and highlight broader temporal trends. This smoothing technique is particularly useful in time-series analysis because environmental measurements such as air pollution often contain substantial short-term variability caused by localized events, measurement noise, or sudden emission spikes.

From the plot, it is evident that the raw AQI values fluctuate significantly throughout the dataset, with numerous short-term spikes that represent sudden increases in pollution levels. These spikes can arise from episodic factors such as traffic congestion, industrial emissions, atmospheric stagnation, or weather events. While these abrupt changes are important for real-time monitoring, they can obscure longer-term patterns when analyzing the data directly. The rolling mean helps address this issue by averaging nearby values and revealing the underlying trend in air quality conditions.

The smoothed AQI curve demonstrates clear periods of gradual increases and decreases over time. Higher AQI levels are visible during certain periods, particularly toward the late months of the year and early months of the following year, while lower AQI levels appear during the middle portion of the timeline. This pattern is consistent with seasonal variations observed in earlier analyses, where atmospheric conditions such as rainfall, wind circulation, and temperature influence pollutant dispersion. During seasons with stronger atmospheric mixing and precipitation, pollutants tend to disperse more effectively, resulting in lower AQI values. Conversely, during periods with more stable atmospheric conditions, pollutant accumulation can lead to elevated AQI levels.

Another important observation is that although the raw AQI values contain sharp peaks, the rolling mean curve changes more gradually. This indicates that many extreme spikes are **short-lived events** rather than persistent pollution episodes. The rolling average therefore provides a more stable indicator of the general air quality conditions over time.

From a modeling perspective, the rolling mean analysis reveals that AQI exhibits **both short-term volatility and longer-term trends**, suggesting that predictive models should account for temporal dependencies at multiple time scales. Short-term lag features may capture sudden pollution spikes, while longer rolling averages can help represent underlying seasonal or trend components. Understanding this structure is essential when designing time-series forecasting models for air quality prediction.


## RNN, LSTM, GRU

In [ ]:
# ============================================================
# AQI Prediction — RNN vs LSTM vs GRU  |  Single Cell
# ============================================================

import numpy as np
import pandas as pd
import warnings, time
warnings.filterwarnings("ignore")

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    mean_absolute_percentage_error, explained_variance_score
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# ── 0. Reproducibility ────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── 1. Load & Inspect ─────────────────────────────────────────
df = pd.read_csv("./cleaned/kiit_weather_cleaned.csv")
df["date_time"] = pd.to_datetime(df["date_time"])
df = df.sort_values("date_time").reset_index(drop=True)

TARGET   = "aqi"
DROP     = ["date_time", "high_aqi"]
FEATURES = [c for c in df.columns if c not in DROP + [TARGET]]

print(f"Dataset shape : {df.shape}")
print(f"Target        : {TARGET}")
print(f"Feature count : {len(FEATURES)}")
print(f"Date range    : {df['date_time'].min()}  →  {df['date_time'].max()}")

# ── 2. Clean ──────────────────────────────────────────────────
df = df[FEATURES + [TARGET]].dropna()
print(f"Shape after dropna : {df.shape}")

# ── 3. Scale ──────────────────────────────────────────────────
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(df[FEATURES].values)
y_scaled = scaler_y.fit_transform(df[[TARGET]].values)

# ── 4. Sequence builder ───────────────────────────────────────
SEQ_LEN = 16

def make_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i : i + seq_len])
        ys.append(y[i + seq_len])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = make_sequences(X_scaled, y_scaled, SEQ_LEN)
print(f"Sequence shape : X={X_seq.shape}  y={y_seq.shape}")

# ── 5. Train / Val / Test split (70 / 15 / 15) ───────────────
n       = len(X_seq)
tr_end  = int(n * 0.70)
val_end = int(n * 0.85)

X_tr,  y_tr  = X_seq[:tr_end],        y_seq[:tr_end]
X_val, y_val = X_seq[tr_end:val_end], y_seq[tr_end:val_end]
X_te,  y_te  = X_seq[val_end:],       y_seq[val_end:]

print(f"Train={len(X_tr)}  Val={len(X_val)}  Test={len(X_te)}")

n_features = X_seq.shape[2]

# ── 6. Model factory ─────────────────────────────────────────
def build_model(kind, units=64, dropout=0.2):
    model = Sequential(name=kind)
    rnn_kwargs = dict(units=units, return_sequences=False)
    if   kind == "RNN":  layer = SimpleRNN(**rnn_kwargs)
    elif kind == "LSTM": layer = LSTM(**rnn_kwargs)
    elif kind == "GRU":  layer = GRU(**rnn_kwargs)
    model.add(layer)
    model.add(Dropout(dropout))
    model.add(Dense(32, activation="relu"))
    model.add(Dense(1))
    model.compile(optimizer=Adam(1e-3), loss="mse", metrics=["mae"])
    return model

# ── 7. Train all three ────────────────────────────────────────
EPOCHS    = 100
BATCH     = 64
callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True, verbose=0),
    ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-6, verbose=0)
]

# ── Cell-scoped results — keyed under "rnn_lstm_gru" ─────────
# Global registry (initialised once across the notebook session)
if "EXPERIMENT_REGISTRY" not in dir(__builtins__) and "EXPERIMENT_REGISTRY" not in globals():
    EXPERIMENT_REGISTRY = {}

_EXP_KEY      = "rnn_lstm_gru"          # ← unique key for THIS cell's experiment
_cell_results  = {}                      # local accumulator, written to registry at the end
_cell_histories = {}

for kind in ["RNN", "LSTM", "GRU"]:
    print(f"\n{'='*50}\nTraining {kind}…")
    t0    = time.time()
    model = build_model(kind)

    hist = model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=EPOCHS, batch_size=BATCH,
        callbacks=callbacks, verbose=0
    )

    elapsed = time.time() - t0
    _cell_histories[kind] = hist.history

    # Predict & inverse-transform
    y_pred_s = model.predict(X_te, verbose=0)
    y_pred   = scaler_y.inverse_transform(y_pred_s).flatten()
    y_true   = scaler_y.inverse_transform(y_te).flatten()

    # ── Metrics ───────────────────────────────────────────────
    mae   = mean_absolute_error(y_true, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    mse   = mean_squared_error(y_true, y_pred)
    r2    = r2_score(y_true, y_pred)
    mape  = mean_absolute_percentage_error(y_true, y_pred) * 100
    evs   = explained_variance_score(y_true, y_pred)
    corr  = np.corrcoef(y_true, y_pred)[0, 1]
    bias  = np.mean(y_pred - y_true)
    max_e = np.max(np.abs(y_pred - y_true))
    std_e = np.std(y_pred - y_true)

    dy_true = np.diff(y_true)
    dy_pred = np.diff(y_pred)
    da      = np.mean(np.sign(dy_true) == np.sign(dy_pred)) * 100

    within10 = np.mean(np.abs(y_pred - y_true) <= 10) * 100
    within20 = np.mean(np.abs(y_pred - y_true) <= 20) * 100

    _cell_results[kind] = dict(
        MAE=mae, RMSE=rmse, MSE=mse, R2=r2, MAPE=mape,
        EVS=evs, Pearson_r=corr, Bias=bias,
        Max_Abs_Err=max_e, Std_Residuals=std_e,
        Directional_Acc=da,
        Within_10AQI=within10, Within_20AQI=within20,
        Train_Time_s=elapsed,
        y_pred=y_pred, y_true=y_true
    )

    epochs_run = len(hist.history["loss"])
    print(f"  Epochs run : {epochs_run}  |  Time : {elapsed:.1f}s")
    print(f"  MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.4f}  MAPE={mape:.2f}%")
    print(f"  Within ±10 AQI : {within10:.1f}%  |  Directional Acc : {da:.1f}%")

# ── Commit to global registry (safe merge, no cross-cell clobber) ──
EXPERIMENT_REGISTRY[_EXP_KEY] = {
    "results":   _cell_results,
    "histories": _cell_histories,
}

# ── 8. Summary table ──────────────────────────────────────────
_metric_cols = [
    "MAE","RMSE","MSE","R2","MAPE","EVS","Pearson_r",
    "Bias","Max_Abs_Err","Std_Residuals",
    "Directional_Acc","Within_10AQI","Within_20AQI","Train_Time_s"
]
_summary = pd.DataFrame(
    {k: {m: _cell_results[k][m] for m in _metric_cols} for k in _cell_results}
).T.round(4)

# Store summary in registry too (handy for cross-experiment comparison cells)
EXPERIMENT_REGISTRY[_EXP_KEY]["summary"] = _summary

print("\n\n━━━━  PERFORMANCE SUMMARY  ━━━━")
print(_summary.to_string())

print("\n━━━━  RANKED BY R²  ━━━━")
print(_summary.sort_values("R2", ascending=False)[
    ["MAE","RMSE","R2","MAPE","Within_10AQI","Directional_Acc","Train_Time_s"]
].to_string())

In [ ]:
# ============================================================
# AQI Plot — RNN vs LSTM vs GRU  (re-run freely)
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# ── Pull ONLY from this experiment's registry slot ───────────
_EXP_KEY    = "rnn_lstm_gru"                              # must match Cell 1
_plot_data  = EXPERIMENT_REGISTRY[_EXP_KEY]["results"]    # no global `results` collision
_plot_hist  = EXPERIMENT_REGISTRY[_EXP_KEY]["histories"]

# ── Customise freely ─────────────────────────────────────────
COLORS   = {"RNN": "#f97316", "LSTM": "#22d3ee", "GRU": "#a78bfa"}
N_SHOW   = 300       # how many test-set points to show in time-series rows
ALPHA_TRUE = 0.6
SAVE_PATH  = "aqi_model_comparison.png"

# ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(22, 18), facecolor="#0d1117")
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# Row 0 — Training loss curves
for i, kind in enumerate(["RNN", "LSTM", "GRU"]):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor("#161b22")
    h = _plot_hist[kind]
    ax.plot(h["loss"],     color=COLORS[kind], lw=1.5, label="Train Loss")
    ax.plot(h["val_loss"], color="white",       lw=1.2, ls="--", label="Val Loss", alpha=0.7)
    ax.set_title(f"{kind} — Loss Curve", color="white", fontsize=11, fontweight="bold")
    ax.set_xlabel("Epoch", color="#8b949e")
    ax.set_ylabel("MSE",   color="#8b949e")
    ax.tick_params(colors="#8b949e")
    ax.legend(fontsize=8)
    for spine in ax.spines.values(): spine.set_edgecolor("#30363d")

# Row 1 — Predicted vs Actual
for i, kind in enumerate(["RNN", "LSTM", "GRU"]):
    ax = fig.add_subplot(gs[1, i])
    ax.set_facecolor("#161b22")
    yt = _plot_data[kind]["y_true"][:N_SHOW]
    yp = _plot_data[kind]["y_pred"][:N_SHOW]
    ax.plot(yt, color="white",      lw=1.0, alpha=ALPHA_TRUE, label="Actual")
    ax.plot(yp, color=COLORS[kind], lw=1.2, label="Predicted")
    ax.set_title(f"{kind} — Actual vs Predicted", color="white", fontsize=11, fontweight="bold")
    ax.set_xlabel("Time steps (test)", color="#8b949e")
    ax.set_ylabel("AQI",              color="#8b949e")
    ax.tick_params(colors="#8b949e")
    ax.legend(fontsize=8)
    for spine in ax.spines.values(): spine.set_edgecolor("#30363d")

# Row 2, Col 0 — Scatter
ax_sc = fig.add_subplot(gs[2, 0])
ax_sc.set_facecolor("#161b22")
for kind in ["RNN", "LSTM", "GRU"]:
    yt = _plot_data[kind]["y_true"]
    yp = _plot_data[kind]["y_pred"]
    ax_sc.scatter(yt, yp, color=COLORS[kind], alpha=0.25, s=6, label=kind)
lims = [min(_plot_data["RNN"]["y_true"].min(), 0),
        _plot_data["RNN"]["y_true"].max() * 1.1]
ax_sc.plot(lims, lims, "w--", lw=1, alpha=0.5)
ax_sc.set_title("Scatter — True vs Pred", color="white", fontsize=11, fontweight="bold")
ax_sc.set_xlabel("True AQI", color="#8b949e")
ax_sc.set_ylabel("Pred AQI", color="#8b949e")
ax_sc.tick_params(colors="#8b949e")
ax_sc.legend(fontsize=8)
for spine in ax_sc.spines.values(): spine.set_edgecolor("#30363d")

# Row 2, Col 1 — Residuals distribution
ax_res = fig.add_subplot(gs[2, 1])
ax_res.set_facecolor("#161b22")
for kind in ["RNN", "LSTM", "GRU"]:
    res = _plot_data[kind]["y_pred"] - _plot_data[kind]["y_true"]
    ax_res.hist(res, bins=50, color=COLORS[kind], alpha=0.5, label=kind, density=True)
ax_res.axvline(0, color="white", lw=1, ls="--", alpha=0.7)
ax_res.set_title("Residuals Distribution", color="white", fontsize=11, fontweight="bold")
ax_res.set_xlabel("Error (Pred − True)", color="#8b949e")
ax_res.set_ylabel("Density",             color="#8b949e")
ax_res.tick_params(colors="#8b949e")
ax_res.legend(fontsize=8)
for spine in ax_res.spines.values(): spine.set_edgecolor("#30363d")

# Row 2, Col 2 — Bar chart of key metrics
ax_bar = fig.add_subplot(gs[2, 2])
ax_bar.set_facecolor("#161b22")
x     = np.arange(3)
kinds = ["RNN", "LSTM", "GRU"]
width = 0.25
for j, metric in enumerate(["MAE", "RMSE", "R2"]):
    vals = [_plot_data[k][metric] for k in kinds]
    ax_bar.bar(x + j * width, vals, width, label=metric,
               color=list(COLORS.values())[j], alpha=0.85)
ax_bar.set_xticks(x + width)
ax_bar.set_xticklabels(kinds, color="white", fontsize=10)
ax_bar.set_title("MAE / RMSE / R² Comparison", color="white", fontsize=11, fontweight="bold")
ax_bar.tick_params(colors="#8b949e")
ax_bar.legend(fontsize=8)
for spine in ax_bar.spines.values(): spine.set_edgecolor("#30363d")

plt.suptitle("AQI Prediction — RNN vs LSTM vs GRU",
             color="white", fontsize=16, fontweight="bold", y=0.98)

plt.savefig(SAVE_PATH, dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print(f"\n✅ Plot saved → {SAVE_PATH}")

## TSAI

In [ ]:
# ============================================================
# AQI Prediction using tsai — LSTMPlus · GRUPlus · InceptionTime · TST
# ============================================================

import subprocess, sys
for pkg in ["tsai", "fastai"]:
    try: __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import numpy as np
import pandas as pd
import warnings, time
warnings.filterwarnings("ignore")

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    mean_absolute_percentage_error, explained_variance_score
)

from tsai.basics import *
from tsai.models.RNNPlus       import LSTMPlus, GRUPlus
from tsai.models.TST           import TST
from tsai.models.InceptionTime import InceptionTime
import fastai
from fastai.callback.tracker import EarlyStoppingCallback, ReduceLROnPlateau

print(f"tsai   version : {tsai.__version__}")
print(f"fastai version : {fastai.__version__}")

# ── 0. Reproducibility ────────────────────────────────────────
SEED = 42
set_seed(SEED, reproducible=True)

# ── 1. Load & Prepare ─────────────────────────────────────────
df = pd.read_csv("./cleaned/kiit_weather_cleaned.csv")
df["date_time"] = pd.to_datetime(df["date_time"])
df = df.sort_values("date_time").reset_index(drop=True)

TARGET   = "aqi"
DROP     = ["date_time", "high_aqi"]
FEATURES = [c for c in df.columns if c not in DROP + [TARGET]]

df = df[FEATURES + [TARGET]].dropna().reset_index(drop=True)
print(f"Dataset shape  : {df.shape}")
print(f"Feature count  : {len(FEATURES)}")

# ── 2. Scale ─────────────────────────────────────────────────
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(df[FEATURES].values).astype(np.float32)
y_scaled = scaler_y.fit_transform(df[[TARGET]].values).astype(np.float32).flatten()

# ── 3. Sliding window → (samples, channels, seq_len) ──────────
SEQ_LEN = 16

def make_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i : i + seq_len])
        ys.append(y[i + seq_len])
    return np.stack(Xs).transpose(0, 2, 1), np.array(ys)

X_seq, y_seq = make_sequences(X_scaled, y_scaled, SEQ_LEN)
print(f"Sequence shape : X={X_seq.shape}  y={y_seq.shape}")

n       = len(X_seq)
tr_end  = int(n * 0.70)
val_end = int(n * 0.85)

splits = (list(range(tr_end)), list(range(tr_end, val_end)))
X_te   = X_seq[val_end:]
y_te   = y_seq[val_end:]

n_features = X_seq.shape[1]

# ── 4. Metric helper ─────────────────────────────────────────
def compute_metrics(y_true, y_pred):
    mae   = mean_absolute_error(y_true, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    mse   = mean_squared_error(y_true, y_pred)
    r2    = r2_score(y_true, y_pred)
    mape  = mean_absolute_percentage_error(y_true, y_pred) * 100
    evs   = explained_variance_score(y_true, y_pred)
    corr  = np.corrcoef(y_true, y_pred)[0, 1]
    bias  = float(np.mean(y_pred - y_true))
    max_e = float(np.max(np.abs(y_pred - y_true)))
    std_e = float(np.std(y_pred - y_true))
    da    = np.mean(np.sign(np.diff(y_true)) == np.sign(np.diff(y_pred))) * 100
    w10   = np.mean(np.abs(y_pred - y_true) <= 10) * 100
    w20   = np.mean(np.abs(y_pred - y_true) <= 20) * 100
    return dict(MAE=mae, RMSE=rmse, MSE=mse, R2=r2, MAPE=mape,
                EVS=evs, Pearson_r=corr, Bias=bias,
                Max_Abs_Err=max_e, Std_Residuals=std_e,
                Directional_Acc=da, Within_10AQI=w10, Within_20AQI=w20)

# ── 5. Model configs ─────────────────────────────────────────
MODEL_CONFIGS = {
    "LSTMPlus": (
        LSTMPlus,
        dict(hidden_size=64, n_layers=2, dropout=0.2),
        1e-3, 50
    ),
    "GRUPlus": (
        GRUPlus,
        dict(hidden_size=64, n_layers=2, dropout=0.2),
        1e-3, 50
    ),
    "InceptionTime": (
        InceptionTime,
        dict(),
        1e-3, 50
    ),
    "TST": (
        TST,
        dict(n_layers=2, d_model=64, n_heads=4, d_ff=128, dropout=0.1),
        3e-4, 50
    ),
}

# ── 6. Train loop ─────────────────────────────────────────────
if "EXPERIMENT_REGISTRY" not in globals():
    EXPERIMENT_REGISTRY = {}

_EXP_KEY        = "tsai_models"       # ← unique key for THIS cell's experiment
_cell_results   = {}
_cell_histories = {}

for _name, (_arch, _arch_kw, _lr, _epochs) in MODEL_CONFIGS.items():
    print(f"\n{'='*55}\nTraining  {_name} …")
    _t0 = time.time()

    _tfms       = [None, TSRegression()]
    _batch_tfms = TSStandardize()

    _fcst = TSForecaster(
        X_seq[:val_end], y_seq[:val_end],
        splits      = splits,
        tfms        = _tfms,
        batch_tfms  = _batch_tfms,
        bs          = 64,
        arch        = _arch,
        arch_config = _arch_kw,
        metrics     = [mae, rmse],
        path        = "tsai_models",
        cbs         = [
            EarlyStoppingCallback(monitor='valid_loss', patience=8),
            ReduceLROnPlateau(monitor='valid_loss', patience=4, min_lr=1e-6)
        ]
    )

    _fcst.fit_one_cycle(_epochs, _lr)

    _elapsed = time.time() - _t0
    _ep_run  = len(_fcst.recorder.values)

    _raw_preds, _, _ = _fcst.get_X_preds(X_te)
    _y_pred_s = np.array(_raw_preds).flatten()

    _y_pred = scaler_y.inverse_transform(_y_pred_s.reshape(-1, 1)).flatten()
    _y_true = scaler_y.inverse_transform(y_te.reshape(-1, 1)).flatten()

    _m = compute_metrics(_y_true, _y_pred)
    _m["Train_Time_s"] = _elapsed
    _m["Epochs_run"]   = _ep_run
    _m["y_pred"]       = _y_pred
    _m["y_true"]       = _y_true
    _cell_results[_name] = _m

    _cell_histories[_name] = {
        "train": [v[0] for v in _fcst.recorder.values],
        "valid": [v[1] for v in _fcst.recorder.values],
    }

    print(f"  Epochs run : {_ep_run}  |  Time : {_elapsed:.1f}s")
    print(f"  MAE={_m['MAE']:.3f}  RMSE={_m['RMSE']:.3f}  "
          f"R²={_m['R2']:.4f}  MAPE={_m['MAPE']:.2f}%")
    print(f"  Within ±10 AQI : {_m['Within_10AQI']:.1f}%  "
          f"Directional Acc : {_m['Directional_Acc']:.1f}%")

# ── Commit to global registry ─────────────────────────────────
_METRIC_COLS = [
    "MAE","RMSE","MSE","R2","MAPE","EVS","Pearson_r",
    "Bias","Max_Abs_Err","Std_Residuals",
    "Directional_Acc","Within_10AQI","Within_20AQI",
    "Train_Time_s","Epochs_run"
]
_summary = pd.DataFrame(
    {k: {m: _cell_results[k][m] for m in _METRIC_COLS} for k in _cell_results}
).T.round(4)

EXPERIMENT_REGISTRY[_EXP_KEY] = {
    "results":   _cell_results,
    "histories": _cell_histories,
    "summary":   _summary,
}

print("\n\n━━━━  TSAI MODEL PERFORMANCE SUMMARY  ━━━━")
print(_summary.to_string())

print("\n━━━━  RANKED BY R²  ━━━━")
print(_summary.sort_values("R2", ascending=False)[
    ["MAE","RMSE","R2","MAPE","Within_10AQI","Directional_Acc","Train_Time_s","Epochs_run"]
].to_string())

In [ ]:
# ============================================================
# AQI Plot — tsai Models  (re-run freely)
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# ── Pull ONLY from this experiment's registry slot ───────────
_EXP_KEY    = "tsai_models"                                # must match Cell 1
_plot_data  = EXPERIMENT_REGISTRY[_EXP_KEY]["results"]
_plot_hist  = EXPERIMENT_REGISTRY[_EXP_KEY]["histories"]

# ── Customise freely ─────────────────────────────────────────
COLORS = {
    "LSTMPlus":      "#f97316",
    "GRUPlus":       "#22d3ee",
    "InceptionTime": "#a78bfa",
    "TST":           "#34d399",
}
N_SHOW    = 300
SAVE_PATH = "aqi_tsai_comparison.png"

# ─────────────────────────────────────────────────────────────
_names         = list(_plot_data.keys())
_names_preview = _names[:3]          # rows 0 & 1 show first 3 models only

fig = plt.figure(figsize=(24, 20), facecolor="#0d1117")
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.48, wspace=0.35)

# Row 0 — Loss curves (first 3 models)
for i, name in enumerate(_names_preview):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor("#161b22")
    h = _plot_hist[name]
    ax.plot(h["train"], color=COLORS[name], lw=1.5, label="Train Loss")
    ax.plot(h["valid"], color="white",      lw=1.2, ls="--", alpha=0.7, label="Val Loss")
    ax.set_title(f"{name} — Loss", color="white", fontsize=10, fontweight="bold")
    ax.set_xlabel("Epoch", color="#8b949e")
    ax.set_ylabel("Loss",  color="#8b949e")
    ax.tick_params(colors="#8b949e")
    ax.legend(fontsize=8)
    for sp in ax.spines.values(): sp.set_edgecolor("#30363d")

# Row 1 — Actual vs Predicted (first 3 models)
for i, name in enumerate(_names_preview):
    ax = fig.add_subplot(gs[1, i])
    ax.set_facecolor("#161b22")
    yt = _plot_data[name]["y_true"][:N_SHOW]
    yp = _plot_data[name]["y_pred"][:N_SHOW]
    ax.plot(yt, color="white",      lw=1.0, alpha=0.6, label="Actual")
    ax.plot(yp, color=COLORS[name], lw=1.2,            label="Predicted")
    ax.set_title(f"{name} — Actual vs Pred", color="white", fontsize=10, fontweight="bold")
    ax.set_xlabel("Test steps", color="#8b949e")
    ax.set_ylabel("AQI",        color="#8b949e")
    ax.tick_params(colors="#8b949e")
    ax.legend(fontsize=8)
    for sp in ax.spines.values(): sp.set_edgecolor("#30363d")

# Row 2, Col 0 — Scatter (all models)
ax_sc = fig.add_subplot(gs[2, 0])
ax_sc.set_facecolor("#161b22")
for name in _names:
    ax_sc.scatter(_plot_data[name]["y_true"], _plot_data[name]["y_pred"],
                  color=COLORS[name], alpha=0.2, s=5, label=name)
lim = [0, max(_plot_data[n]["y_true"].max() for n in _names) * 1.05]
ax_sc.plot(lim, lim, "w--", lw=1, alpha=0.5)
ax_sc.set_title("True vs Predicted (all models)", color="white", fontsize=10, fontweight="bold")
ax_sc.set_xlabel("True AQI",  color="#8b949e")
ax_sc.set_ylabel("Pred AQI",  color="#8b949e")
ax_sc.tick_params(colors="#8b949e")
ax_sc.legend(fontsize=7, markerscale=2)
for sp in ax_sc.spines.values(): sp.set_edgecolor("#30363d")

# Row 2, Col 1 — Residuals (all models)
ax_r = fig.add_subplot(gs[2, 1])
ax_r.set_facecolor("#161b22")
for name in _names:
    res = _plot_data[name]["y_pred"] - _plot_data[name]["y_true"]
    ax_r.hist(res, bins=50, color=COLORS[name], alpha=0.45, label=name, density=True)
ax_r.axvline(0, color="white", lw=1, ls="--", alpha=0.7)
ax_r.set_title("Residuals Distribution", color="white", fontsize=10, fontweight="bold")
ax_r.set_xlabel("Error (Pred − True)", color="#8b949e")
ax_r.set_ylabel("Density",             color="#8b949e")
ax_r.tick_params(colors="#8b949e")
ax_r.legend(fontsize=7)
for sp in ax_r.spines.values(): sp.set_edgecolor("#30363d")

# Row 2, Col 2 — Metric bars (all models)
ax_b = fig.add_subplot(gs[2, 2])
ax_b.set_facecolor("#161b22")
x      = np.arange(len(_names))
width  = 0.25
bar_c  = ["#f97316", "#22d3ee", "#a78bfa"]
for j, metric in enumerate(["MAE", "RMSE", "R2"]):
    vals = [_plot_data[k][metric] for k in _names]
    ax_b.bar(x + j * width, vals, width, label=metric, color=bar_c[j], alpha=0.85)
ax_b.set_xticks(x + width)
ax_b.set_xticklabels(_names, color="white", fontsize=8, rotation=20, ha="right")
ax_b.set_title("MAE / RMSE / R² Comparison", color="white", fontsize=10, fontweight="bold")
ax_b.tick_params(colors="#8b949e")
ax_b.legend(fontsize=8)
for sp in ax_b.spines.values(): sp.set_edgecolor("#30363d")

plt.suptitle(
    "AQI Prediction with tsai  —  LSTMPlus · GRUPlus · InceptionTime · TST",
    color="white", fontsize=14, fontweight="bold", y=0.99
)
plt.savefig(SAVE_PATH, dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print(f"\n✅ Plot saved → {SAVE_PATH}")

## Classic Neural Networks

In [ ]:
# ============================================================
# AQI Prediction — Classic Neural Nets (PyTorch / tsai)
# DeepMLP · ResNet1D · FCN · TCN · WaveNet · NBEATS
# ============================================================

import subprocess, sys
for pkg in ["tsai", "fastai"]:
    try: __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import numpy as np
import pandas as pd
import warnings, time
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    mean_absolute_percentage_error, explained_variance_score
)

from tsai.basics import *
from tsai.models.ResNet import ResNet
from tsai.models.FCN    import FCN
from tsai.models.TCN    import TCN
import fastai
from fastai.callback.tracker import EarlyStoppingCallback, ReduceLROnPlateau

print(f"tsai   version : {tsai.__version__}")
print(f"fastai version : {fastai.__version__}")
print(f"torch  version : {torch.__version__}")

# ── 0. Reproducibility ────────────────────────────────────────
SEED = 42
set_seed(SEED, reproducible=True)

# ── 1. Load & Prepare ─────────────────────────────────────────
df = pd.read_csv("./cleaned/kiit_weather_cleaned.csv")
df["date_time"] = pd.to_datetime(df["date_time"])
df = df.sort_values("date_time").reset_index(drop=True)

TARGET   = "aqi"
DROP     = ["date_time", "high_aqi"]
FEATURES = [c for c in df.columns if c not in DROP + [TARGET]]

df = df[FEATURES + [TARGET]].dropna().reset_index(drop=True)
print(f"Dataset shape  : {df.shape}")
print(f"Feature count  : {len(FEATURES)}")

# ── 2. Scale ─────────────────────────────────────────────────
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(df[FEATURES].values).astype(np.float32)
y_scaled = scaler_y.fit_transform(df[[TARGET]].values).astype(np.float32).flatten()

# ── 3. Sliding window → (samples, channels, seq_len) ──────────
SEQ_LEN = 16

def make_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i : i + seq_len])
        ys.append(y[i + seq_len])
    return np.stack(Xs).transpose(0, 2, 1), np.array(ys)

X_seq, y_seq = make_sequences(X_scaled, y_scaled, SEQ_LEN)
print(f"Sequence shape : X={X_seq.shape}  y={y_seq.shape}")

n       = len(X_seq)
tr_end  = int(n * 0.70)
val_end = int(n * 0.85)

splits = (list(range(tr_end)), list(range(tr_end, val_end)))
X_te   = X_seq[val_end:]
y_te   = y_seq[val_end:]

n_features = X_seq.shape[1]

# ── 4. Custom architectures ───────────────────────────────────

class DeepMLP(nn.Module):
    def __init__(self, c_in, seq_len, hidden=(256, 128, 64), dropout=0.3):
        super().__init__()
        in_dim = c_in * seq_len
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                       nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x.flatten(1)).squeeze(-1)


class CausalConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, dilation=1):
        super().__init__()
        self.pad  = (kernel - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel,
                              dilation=dilation, padding=self.pad)

    def forward(self, x):
        return self.conv(x)[..., :-self.pad] if self.pad else self.conv(x)

class WaveNetBlock(nn.Module):
    def __init__(self, ch, kernel=3, dilation=1):
        super().__init__()
        self.tanh_conv = CausalConv1d(ch, ch, kernel, dilation)
        self.sig_conv  = CausalConv1d(ch, ch, kernel, dilation)
        self.res_conv  = nn.Conv1d(ch, ch, 1)
        self.skip_conv = nn.Conv1d(ch, ch, 1)

    def forward(self, x):
        gate = torch.tanh(self.tanh_conv(x)) * torch.sigmoid(self.sig_conv(x))
        return self.res_conv(gate) + x, self.skip_conv(gate)

class WaveNetRegressor(nn.Module):
    def __init__(self, c_in, hidden=32, n_blocks=6):
        super().__init__()
        self.input_conv = nn.Conv1d(c_in, hidden, 1)
        dilations = [2**i for i in range(n_blocks)]
        self.blocks = nn.ModuleList(
            [WaveNetBlock(hidden, kernel=2, dilation=d) for d in dilations]
        )
        self.head = nn.Sequential(
            nn.ReLU(), nn.Conv1d(hidden, hidden, 1),
            nn.ReLU(), nn.Conv1d(hidden, 1, 1)
        )

    def forward(self, x):
        x = self.input_conv(x)
        skip_sum = 0
        for block in self.blocks:
            x, skip = block(x)
            skip_sum = skip_sum + skip
        return self.head(skip_sum)[..., -1].squeeze(-1)


class NBEATSBlock(nn.Module):
    def __init__(self, in_dim, theta_dim=32, hidden=128, layers=4):
        super().__init__()
        fc = []
        prev = in_dim
        for _ in range(layers):
            fc += [nn.Linear(prev, hidden), nn.ReLU()]
            prev = hidden
        self.fc        = nn.Sequential(*fc)
        self.theta_b   = nn.Linear(hidden, theta_dim)
        self.theta_f   = nn.Linear(hidden, theta_dim)
        self.back_proj = nn.Linear(theta_dim, in_dim)
        self.fore_proj = nn.Linear(theta_dim, 1)

    def forward(self, x):
        h        = self.fc(x)
        backcast = self.back_proj(self.theta_b(h))
        forecast = self.fore_proj(self.theta_f(h))
        return backcast, forecast

class NBEATSRegressor(nn.Module):
    def __init__(self, c_in, seq_len, n_stacks=3, theta_dim=32,
                 hidden=128, block_layers=4):
        super().__init__()
        in_dim = c_in * seq_len
        self.stacks = nn.ModuleList(
            [NBEATSBlock(in_dim, theta_dim, hidden, block_layers)
             for _ in range(n_stacks)]
        )

    def forward(self, x):
        residual = x.flatten(1)
        forecast = 0
        for stack in self.stacks:
            backcast, f = stack(residual)
            residual    = residual - backcast
            forecast    = forecast + f
        return forecast.squeeze(-1)


# ── 5. TSForecaster-compatible factory functions ──────────────
def make_deep_mlp(c_in, c_out):
    return DeepMLP(c_in, SEQ_LEN, hidden=(256, 128, 64), dropout=0.3)

def make_wavenet(c_in, c_out):
    return WaveNetRegressor(c_in, hidden=32, n_blocks=6)

def make_nbeats(c_in, c_out):
    return NBEATSRegressor(c_in, SEQ_LEN, n_stacks=3, theta_dim=32, hidden=128)


# ── 6. Model configs ──────────────────────────────────────────
MODEL_CONFIGS = {
    "DeepMLP":  (make_deep_mlp,  {},             1e-3, 60),
    "ResNet1D": (ResNet,         {},             1e-3, 60),
    "FCN":      (FCN,            {},             1e-3, 60),
    "TCN":      (TCN,            dict(ks=3),     5e-4, 60),
    "WaveNet":  (make_wavenet,   {},             5e-4, 60),
    "NBEATS":   (make_nbeats,    {},             1e-3, 60),
}

# ── 7. Metric helper ─────────────────────────────────────────
def compute_metrics(y_true, y_pred):
    mae   = mean_absolute_error(y_true, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    mse   = mean_squared_error(y_true, y_pred)
    r2    = r2_score(y_true, y_pred)
    mape  = mean_absolute_percentage_error(y_true, y_pred) * 100
    evs   = explained_variance_score(y_true, y_pred)
    corr  = np.corrcoef(y_true, y_pred)[0, 1]
    bias  = float(np.mean(y_pred - y_true))
    max_e = float(np.max(np.abs(y_pred - y_true)))
    std_e = float(np.std(y_pred - y_true))
    da    = np.mean(np.sign(np.diff(y_true)) == np.sign(np.diff(y_pred))) * 100
    w10   = np.mean(np.abs(y_pred - y_true) <= 10) * 100
    w20   = np.mean(np.abs(y_pred - y_true) <= 20) * 100
    return dict(MAE=mae, RMSE=rmse, MSE=mse, R2=r2, MAPE=mape,
                EVS=evs, Pearson_r=corr, Bias=bias,
                Max_Abs_Err=max_e, Std_Residuals=std_e,
                Directional_Acc=da, Within_10AQI=w10, Within_20AQI=w20)

# ── 8. Train loop ─────────────────────────────────────────────
if "EXPERIMENT_REGISTRY" not in globals():
    EXPERIMENT_REGISTRY = {}

_EXP_KEY        = "classic_nn"        # ← unique key for THIS cell's experiment
_cell_results   = {}
_cell_histories = {}

for _name, (_arch, _arch_kw, _lr, _epochs) in MODEL_CONFIGS.items():
    print(f"\n{'='*55}\nTraining  {_name} …")
    _t0 = time.time()

    _tfms       = [None, TSRegression()]
    _batch_tfms = TSStandardize()

    _fcst = TSForecaster(
        X_seq[:val_end], y_seq[:val_end],
        splits      = splits,
        tfms        = _tfms,
        batch_tfms  = _batch_tfms,
        bs          = 64,
        arch        = _arch,
        arch_config = _arch_kw,
        metrics     = [mae, rmse],
        path        = "tsai_models",
        cbs         = [
            EarlyStoppingCallback(monitor='valid_loss', patience=8),
            ReduceLROnPlateau(monitor='valid_loss', patience=4, min_lr=1e-6)
        ]
    )

    _fcst.fit_one_cycle(_epochs, _lr)

    _elapsed = time.time() - _t0
    _ep_run  = len(_fcst.recorder.values)

    _raw_preds, _, _ = _fcst.get_X_preds(X_te)
    _y_pred_s = np.array(_raw_preds).flatten()

    _y_pred = scaler_y.inverse_transform(_y_pred_s.reshape(-1, 1)).flatten()
    _y_true = scaler_y.inverse_transform(y_te.reshape(-1, 1)).flatten()

    _m = compute_metrics(_y_true, _y_pred)
    _m["Train_Time_s"] = _elapsed
    _m["Epochs_run"]   = _ep_run
    _m["y_pred"]       = _y_pred
    _m["y_true"]       = _y_true
    _cell_results[_name] = _m

    _cell_histories[_name] = {
        "train": [v[0] for v in _fcst.recorder.values],
        "valid": [v[1] for v in _fcst.recorder.values],
    }

    print(f"  Epochs run : {_ep_run}  |  Time : {_elapsed:.1f}s")
    print(f"  MAE={_m['MAE']:.3f}  RMSE={_m['RMSE']:.3f}  "
          f"R²={_m['R2']:.4f}  MAPE={_m['MAPE']:.2f}%")
    print(f"  Within ±10 AQI : {_m['Within_10AQI']:.1f}%  "
          f"Directional Acc : {_m['Directional_Acc']:.1f}%")

# ── Commit to global registry ─────────────────────────────────
_METRIC_COLS = [
    "MAE","RMSE","MSE","R2","MAPE","EVS","Pearson_r",
    "Bias","Max_Abs_Err","Std_Residuals",
    "Directional_Acc","Within_10AQI","Within_20AQI",
    "Train_Time_s","Epochs_run"
]
_summary = pd.DataFrame(
    {k: {m: _cell_results[k][m] for m in _METRIC_COLS} for k in _cell_results}
).T.round(4)

EXPERIMENT_REGISTRY[_EXP_KEY] = {
    "results":   _cell_results,
    "histories": _cell_histories,
    "summary":   _summary,
}

print("\n\n━━━━  CLASSIC NET MODEL PERFORMANCE SUMMARY  ━━━━")
print(_summary.to_string())

print("\n━━━━  RANKED BY R²  ━━━━")
print(_summary.sort_values("R2", ascending=False)[
    ["MAE","RMSE","R2","MAPE","Within_10AQI","Directional_Acc","Train_Time_s","Epochs_run"]
].to_string())

In [ ]:
# ============================================================
# AQI Plot — Classic Neural Nets  (re-run freely)
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# ── Pull ONLY from this experiment's registry slot ───────────
_EXP_KEY    = "classic_nn"                                 # must match Cell 1
_plot_data  = EXPERIMENT_REGISTRY[_EXP_KEY]["results"]
_plot_hist  = EXPERIMENT_REGISTRY[_EXP_KEY]["histories"]

# ── Customise freely ─────────────────────────────────────────
COLORS = {
    "DeepMLP":  "#f97316",
    "ResNet1D": "#22d3ee",
    "FCN":      "#a78bfa",
    "TCN":      "#34d399",
    "WaveNet":  "#fb7185",
    "NBEATS":   "#facc15",
}
N_SHOW    = 300
SAVE_PATH = "aqi_classic_nn_comparison.png"

# ─────────────────────────────────────────────────────────────
_names         = list(_plot_data.keys())
_names_preview = _names[:3]          # rows 0 & 1 show first 3 models only

fig = plt.figure(figsize=(26, 22), facecolor="#0d1117")
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.35)

# Row 0 — Loss curves (first 3 models)
for i, name in enumerate(_names_preview):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor("#161b22")
    h = _plot_hist[name]
    ax.plot(h["train"], color=COLORS[name], lw=1.5, label="Train Loss")
    ax.plot(h["valid"], color="white",      lw=1.2, ls="--", alpha=0.7, label="Val Loss")
    ax.set_title(f"{name} — Loss", color="white", fontsize=10, fontweight="bold")
    ax.set_xlabel("Epoch", color="#8b949e")
    ax.set_ylabel("Loss",  color="#8b949e")
    ax.tick_params(colors="#8b949e")
    ax.legend(fontsize=8)
    for sp in ax.spines.values(): sp.set_edgecolor("#30363d")

# Row 1 — Actual vs Predicted (first 3 models)
for i, name in enumerate(_names_preview):
    ax = fig.add_subplot(gs[1, i])
    ax.set_facecolor("#161b22")
    yt = _plot_data[name]["y_true"][:N_SHOW]
    yp = _plot_data[name]["y_pred"][:N_SHOW]
    ax.plot(yt, color="white",      lw=1.0, alpha=0.6, label="Actual")
    ax.plot(yp, color=COLORS[name], lw=1.2,            label="Predicted")
    ax.set_title(f"{name} — Actual vs Pred", color="white", fontsize=10, fontweight="bold")
    ax.set_xlabel("Test steps", color="#8b949e")
    ax.set_ylabel("AQI",        color="#8b949e")
    ax.tick_params(colors="#8b949e")
    ax.legend(fontsize=8)
    for sp in ax.spines.values(): sp.set_edgecolor("#30363d")

# Row 2, Col 0 — Scatter (all models)
ax_sc = fig.add_subplot(gs[2, 0])
ax_sc.set_facecolor("#161b22")
for name in _names:
    ax_sc.scatter(_plot_data[name]["y_true"], _plot_data[name]["y_pred"],
                  color=COLORS[name], alpha=0.18, s=5, label=name)
lim = [0, max(_plot_data[n]["y_true"].max() for n in _names) * 1.05]
ax_sc.plot(lim, lim, "w--", lw=1, alpha=0.5)
ax_sc.set_title("True vs Predicted — all models", color="white", fontsize=10, fontweight="bold")
ax_sc.set_xlabel("True AQI",  color="#8b949e")
ax_sc.set_ylabel("Pred AQI",  color="#8b949e")
ax_sc.tick_params(colors="#8b949e")
ax_sc.legend(fontsize=7, markerscale=2)
for sp in ax_sc.spines.values(): sp.set_edgecolor("#30363d")

# Row 2, Col 1 — Residuals (all models)
ax_r = fig.add_subplot(gs[2, 1])
ax_r.set_facecolor("#161b22")
for name in _names:
    res = _plot_data[name]["y_pred"] - _plot_data[name]["y_true"]
    ax_r.hist(res, bins=50, color=COLORS[name], alpha=0.40, label=name, density=True)
ax_r.axvline(0, color="white", lw=1, ls="--", alpha=0.7)
ax_r.set_title("Residuals Distribution", color="white", fontsize=10, fontweight="bold")
ax_r.set_xlabel("Error (Pred − True)", color="#8b949e")
ax_r.set_ylabel("Density",             color="#8b949e")
ax_r.tick_params(colors="#8b949e")
ax_r.legend(fontsize=7)
for sp in ax_r.spines.values(): sp.set_edgecolor("#30363d")

# Row 2, Col 2 — Metric bars (all models)
ax_b = fig.add_subplot(gs[2, 2])
ax_b.set_facecolor("#161b22")
x      = np.arange(len(_names))
width  = 0.25
bar_c  = ["#f97316", "#22d3ee", "#a78bfa"]
for j, metric in enumerate(["MAE", "RMSE", "R2"]):
    vals = [_plot_data[k][metric] for k in _names]
    ax_b.bar(x + j * width, vals, width, label=metric, color=bar_c[j], alpha=0.85)
ax_b.set_xticks(x + width)
ax_b.set_xticklabels(_names, color="white", fontsize=8, rotation=25, ha="right")
ax_b.set_title("MAE / RMSE / R² Comparison", color="white", fontsize=10, fontweight="bold")
ax_b.tick_params(colors="#8b949e")
ax_b.legend(fontsize=8)
for sp in ax_b.spines.values(): sp.set_edgecolor("#30363d")

plt.suptitle(
    "AQI Prediction — Classic Neural Nets: DeepMLP · ResNet1D · FCN · TCN · WaveNet · NBEATS",
    color="white", fontsize=13, fontweight="bold", y=0.99
)
plt.savefig(SAVE_PATH, dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print(f"\n✅  Plot saved → {SAVE_PATH}")

## LNN

In [ ]:
# ============================================================
# AQI Prediction — Liquid / Continuous-Time Neural Networks
# LTC · CfC · NCP-LTC · NCP-CfC
# ============================================================

import numpy as np
import pandas as pd
import warnings, time
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from ncps.torch   import CfC, LTC
from ncps.wirings import AutoNCP, FullyConnected

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    mean_absolute_percentage_error, explained_variance_score
)

print(f"PyTorch : {torch.__version__}")
import ncps; print(f"ncps    : {ncps.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {DEVICE}")

# ── 0. Reproducibility ────────────────────────────────────────
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

# ── 1. Load & Prepare ─────────────────────────────────────────
df = pd.read_csv("./cleaned/kiit_weather_cleaned.csv")
df["date_time"] = pd.to_datetime(df["date_time"])
df = df.sort_values("date_time").reset_index(drop=True)

TARGET   = "aqi"
DROP     = ["date_time", "high_aqi"]
FEATURES = [c for c in df.columns if c not in DROP + [TARGET]]
df       = df[FEATURES + [TARGET]].dropna().reset_index(drop=True)

print(f"Dataset shape : {df.shape}  |  Features : {len(FEATURES)}")

# ── 2. Scale ─────────────────────────────────────────────────
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
X_sc = scaler_X.fit_transform(df[FEATURES].values).astype(np.float32)
y_sc = scaler_y.fit_transform(df[[TARGET]].values).astype(np.float32).flatten()

# ── 3. Sliding-window sequences ───────────────────────────────
SEQ_LEN = 16

def make_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i : i + seq_len])
        ys.append(y[i + seq_len])
    return np.array(Xs), np.array(ys).reshape(-1, 1)

X_seq, y_seq = make_sequences(X_sc, y_sc, SEQ_LEN)
print(f"Sequence shape : X={X_seq.shape}  y={y_seq.shape}")

n = len(X_seq)
tr_end, val_end = int(n * 0.70), int(n * 0.85)

X_tr,  y_tr  = X_seq[:tr_end],        y_seq[:tr_end]
X_val, y_val = X_seq[tr_end:val_end], y_seq[tr_end:val_end]
X_te,  y_te  = X_seq[val_end:],       y_seq[val_end:]

def to_tensor(*arrs): return [torch.tensor(a).to(DEVICE) for a in arrs]

Xtr, ytr   = to_tensor(X_tr,  y_tr)
Xval, yval = to_tensor(X_val, y_val)
Xte, yte   = to_tensor(X_te,  y_te)

train_dl = DataLoader(TensorDataset(Xtr, ytr),   batch_size=64, shuffle=False)
val_dl   = DataLoader(TensorDataset(Xval, yval), batch_size=64, shuffle=False)

n_features = X_seq.shape[2]

# ── 4. Model definitions ──────────────────────────────────────
UNITS       = 64
NCP_NEURONS = 64

class LiquidNet(nn.Module):
    def __init__(self, rnn_cell, hidden_size):
        super().__init__()
        self.rnn  = rnn_cell
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        out, _ = self.rnn(x)
        if out.dim() == 3:
            out = out[:, -1, :]
        return self.head(out)

class LiquidNetNCP(nn.Module):
    def __init__(self, rnn_cell):
        super().__init__()
        self.rnn = rnn_cell

    def forward(self, x):
        out, _ = self.rnn(x)
        if out.dim() == 3:
            out = out[:, -1, :]
        return out   # already (batch, 1) from AutoNCP output_size=1

# ── 5. Metric helper ─────────────────────────────────────────
def compute_metrics(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mse  = mean_squared_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    evs  = explained_variance_score(y_true, y_pred)
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    bias = float(np.mean(y_pred - y_true))
    maxe = float(np.max(np.abs(y_pred - y_true)))
    std  = float(np.std(y_pred - y_true))
    da   = np.mean(np.sign(np.diff(y_true)) == np.sign(np.diff(y_pred))) * 100
    w10  = np.mean(np.abs(y_pred - y_true) <= 10) * 100
    w20  = np.mean(np.abs(y_pred - y_true) <= 20) * 100
    return dict(MAE=mae, RMSE=rmse, MSE=mse, R2=r2, MAPE=mape,
                EVS=evs, Pearson_r=corr, Bias=bias,
                Max_Abs_Err=maxe, Std_Residuals=std,
                Directional_Acc=da, Within_10AQI=w10, Within_20AQI=w20)

# ── 6. Training function ──────────────────────────────────────
def train_model(model, train_dl, val_dl, epochs=80, lr=1e-3, patience=10):
    model.to(DEVICE)
    opt       = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    opt, factor=0.5, patience=5, min_lr=1e-6)
    criterion = nn.MSELoss()

    best_val, best_state, wait = np.inf, None, 0
    train_losses, val_losses   = [], []

    for epoch in range(1, epochs + 1):
        model.train()
        batch_losses = []
        for xb, yb in train_dl:
            opt.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            batch_losses.append(loss.item())
        tr_loss = np.mean(batch_losses)

        model.eval()
        with torch.no_grad():
            vl_preds = [model(xb) for xb, _ in val_dl]
            vl_true  = [yb        for _,  yb in val_dl]
            val_loss = criterion(torch.cat(vl_preds), torch.cat(vl_true)).item()

        train_losses.append(tr_loss)
        val_losses.append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val - 1e-6:
            best_val   = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait       = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"    Early stop at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    return train_losses, val_losses

# ── 7. Build, train, evaluate all models ─────────────────────
MODEL_DEFS = {
    "LTC":     lambda: LiquidNet(
                   LTC(n_features, FullyConnected(UNITS, output_dim=UNITS),
                       return_sequences=True), UNITS),
    "CfC":     lambda: LiquidNet(
                   CfC(n_features, UNITS, return_sequences=True), UNITS),
    "NCP-LTC": lambda: LiquidNetNCP(
                   LTC(n_features, AutoNCP(NCP_NEURONS, output_size=1),
                       return_sequences=False)),
    "NCP-CfC": lambda: LiquidNetNCP(
                   CfC(n_features, AutoNCP(NCP_NEURONS, output_size=1),
                       return_sequences=False)),
}

# ── Cell-scoped results — keyed under "liquid_nn" ────────────
if "EXPERIMENT_REGISTRY" not in globals():
    EXPERIMENT_REGISTRY = {}

_EXP_KEY         = "liquid_nn"        # ← unique key for THIS cell's experiment
_cell_results    = {}
_cell_histories  = {}

for _name, _build_fn in MODEL_DEFS.items():
    print(f"\n{'='*55}\nTraining  {_name} …")
    _model    = _build_fn()
    _n_params = sum(p.numel() for p in _model.parameters() if p.requires_grad)
    print(f"  Trainable params : {_n_params:,}")

    _t0 = time.time()
    _tr_l, _val_l = train_model(_model, train_dl, val_dl,
                                 epochs=80, lr=1e-3, patience=10)
    _elapsed = time.time() - _t0

    _cell_histories[_name] = {"train": _tr_l, "val": _val_l}

    _model.eval()
    with torch.no_grad():
        _raw = _model(Xte).cpu().numpy()

    _y_pred = scaler_y.inverse_transform(_raw).flatten()
    _y_true = scaler_y.inverse_transform(yte.cpu().numpy()).flatten()

    _m = compute_metrics(_y_true, _y_pred)
    _m["Train_Time_s"] = _elapsed
    _m["Epochs_run"]   = len(_tr_l)
    _m["Params"]       = _n_params
    _m["y_pred"]       = _y_pred
    _m["y_true"]       = _y_true
    _cell_results[_name] = _m

    print(f"  Epochs run : {len(_tr_l)}  |  Time : {_elapsed:.1f}s")
    print(f"  MAE={_m['MAE']:.3f}  RMSE={_m['RMSE']:.3f}  "
          f"R²={_m['R2']:.4f}  MAPE={_m['MAPE']:.2f}%")
    print(f"  Within ±10 AQI : {_m['Within_10AQI']:.1f}%  "
          f"Directional Acc : {_m['Directional_Acc']:.1f}%")

# ── Commit to global registry ─────────────────────────────────
_METRIC_COLS = [
    "MAE","RMSE","MSE","R2","MAPE","EVS","Pearson_r",
    "Bias","Max_Abs_Err","Std_Residuals",
    "Directional_Acc","Within_10AQI","Within_20AQI",
    "Train_Time_s","Epochs_run","Params"
]
_summary = pd.DataFrame(
    {k: {m: _cell_results[k][m] for m in _METRIC_COLS} for k in _cell_results}
).T.round(4)

EXPERIMENT_REGISTRY[_EXP_KEY] = {
    "results":   _cell_results,
    "histories": _cell_histories,
    "summary":   _summary,
}

print("\n\n━━━━  LIQUID NEURAL NETWORK PERFORMANCE SUMMARY  ━━━━")
print(_summary.to_string())

print("\n━━━━  RANKED BY R²  ━━━━")
print(_summary.sort_values("R2", ascending=False)[
    ["MAE","RMSE","R2","MAPE","Within_10AQI","Directional_Acc","Params","Train_Time_s"]
].to_string())

In [ ]:
# ============================================================
# AQI Plot — Liquid Neural Networks  (re-run freely)
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# ── Pull ONLY from this experiment's registry slot ───────────
_EXP_KEY    = "liquid_nn"                                  # must match Cell 1
_plot_data  = EXPERIMENT_REGISTRY[_EXP_KEY]["results"]
_plot_hist  = EXPERIMENT_REGISTRY[_EXP_KEY]["histories"]

# ── Customise freely ─────────────────────────────────────────
COLORS = {
    "LTC":     "#f97316",
    "CfC":     "#22d3ee",
    "NCP-LTC": "#a78bfa",
    "NCP-CfC": "#34d399",
}
N_SHOW    = 300
SAVE_PATH = "aqi_liquid_nn_comparison.png"

# ─────────────────────────────────────────────────────────────
_names = list(_plot_data.keys())

fig = plt.figure(figsize=(24, 20), facecolor="#0d1117")
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.48, wspace=0.38)

# Row 0 — Loss curves
for i, name in enumerate(_names):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor("#161b22")
    h = _plot_hist[name]
    ax.plot(h["train"], color=COLORS[name], lw=1.5, label="Train")
    ax.plot(h["val"],   color="white",      lw=1.2, ls="--", alpha=0.7, label="Val")
    ax.set_title(f"{name}\nLoss Curve", color="white", fontsize=9, fontweight="bold")
    ax.set_xlabel("Epoch", color="#8b949e", fontsize=8)
    ax.set_ylabel("MSE",   color="#8b949e", fontsize=8)
    ax.tick_params(colors="#8b949e", labelsize=7)
    ax.legend(fontsize=7)
    for sp in ax.spines.values(): sp.set_edgecolor("#30363d")

# Row 1 — Actual vs Predicted
for i, name in enumerate(_names):
    ax = fig.add_subplot(gs[1, i])
    ax.set_facecolor("#161b22")
    yt = _plot_data[name]["y_true"][:N_SHOW]
    yp = _plot_data[name]["y_pred"][:N_SHOW]
    ax.plot(yt, color="white",      lw=1.0, alpha=0.55, label="Actual")
    ax.plot(yp, color=COLORS[name], lw=1.2,             label="Predicted")
    ax.set_title(f"{name}\nActual vs Pred", color="white", fontsize=9, fontweight="bold")
    ax.set_xlabel("Test steps", color="#8b949e", fontsize=8)
    ax.set_ylabel("AQI",        color="#8b949e", fontsize=8)
    ax.tick_params(colors="#8b949e", labelsize=7)
    ax.legend(fontsize=7)
    for sp in ax.spines.values(): sp.set_edgecolor("#30363d")

# Row 2, Col 0-1 — Scatter (merged)
ax_sc = fig.add_subplot(gs[2, 0:2])
ax_sc.set_facecolor("#161b22")
for name in _names:
    ax_sc.scatter(_plot_data[name]["y_true"], _plot_data[name]["y_pred"],
                  color=COLORS[name], alpha=0.2, s=5, label=name)
lim = [0, max(_plot_data[n]["y_true"].max() for n in _names) * 1.05]
ax_sc.plot(lim, lim, "w--", lw=1, alpha=0.5)
ax_sc.set_title("True vs Predicted — All Models", color="white", fontsize=10, fontweight="bold")
ax_sc.set_xlabel("True AQI",  color="#8b949e")
ax_sc.set_ylabel("Pred AQI",  color="#8b949e")
ax_sc.tick_params(colors="#8b949e")
ax_sc.legend(fontsize=8, markerscale=3)
for sp in ax_sc.spines.values(): sp.set_edgecolor("#30363d")

# Row 2, Col 2 — Residuals
ax_r = fig.add_subplot(gs[2, 2])
ax_r.set_facecolor("#161b22")
for name in _names:
    res = _plot_data[name]["y_pred"] - _plot_data[name]["y_true"]
    ax_r.hist(res, bins=50, color=COLORS[name], alpha=0.45, label=name, density=True)
ax_r.axvline(0, color="white", lw=1, ls="--", alpha=0.7)
ax_r.set_title("Residuals Distribution", color="white", fontsize=10, fontweight="bold")
ax_r.set_xlabel("Error (Pred − True)", color="#8b949e")
ax_r.set_ylabel("Density",             color="#8b949e")
ax_r.tick_params(colors="#8b949e")
ax_r.legend(fontsize=7)
for sp in ax_r.spines.values(): sp.set_edgecolor("#30363d")

# Row 2, Col 3 — Metric bars
ax_b = fig.add_subplot(gs[2, 3])
ax_b.set_facecolor("#161b22")
x         = np.arange(len(_names))
width     = 0.25
bar_colors = ["#f97316", "#22d3ee", "#a78bfa"]
for j, metric in enumerate(["MAE", "RMSE", "R2"]):
    vals = [_plot_data[k][metric] for k in _names]
    ax_b.bar(x + j * width, vals, width, label=metric,
             color=bar_colors[j], alpha=0.85)
ax_b.set_xticks(x + width)
ax_b.set_xticklabels(_names, color="white", fontsize=8, rotation=15, ha="right")
ax_b.set_title("MAE / RMSE / R²", color="white", fontsize=10, fontweight="bold")
ax_b.tick_params(colors="#8b949e")
ax_b.legend(fontsize=8)
for sp in ax_b.spines.values(): sp.set_edgecolor("#30363d")

plt.suptitle(
    "AQI Prediction — Liquid Neural Networks via python-ncps\n"
    "LTC  ·  CfC  ·  NCP-LTC  ·  NCP-CfC",
    color="white", fontsize=13, fontweight="bold", y=0.99
)
plt.savefig(SAVE_PATH, dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print(f"\n✅ Plot saved → {SAVE_PATH}")

## Experiment Registery format

In [ ]:
# EXPERIMENT_REGISTRY = {
#     "rnn_lstm_gru":  {...},   # RNN, LSTM, GRU          (keras)
#     "liquid_nn":     {...},   # LTC, CfC, NCP-LTC, NCP-CfC  (ncps/torch)
#     "classic_nn":    {...},   # DeepMLP, ResNet1D, FCN, TCN, WaveNet, NBEATS (tsai)
#     "tsai_models":   {...},   # LSTMPlus, GRUPlus, InceptionTime, TST (tsai)
# }

# # Grand unified comparison — all 17 models in one shot
# _all = {exp: EXPERIMENT_REGISTRY[exp]["results"] for exp in EXPERIMENT_REGISTRY}